# Class-conditional conformal prediction with data augmentation:  campaign

Runs the full simulation at R = 300. Design decisions were fixed in
`pilot_colab.ipynb`; results are read in `analysis.ipynb`.

**Colab disconnects.** The session drops after roughly 90 minutes of browser
inactivity and after 12 hours regardless. This notebook is built around that:

- results go to Google Drive, never to `/content`, which is wiped;
- a checkpoint is written every 10 replications, so a disconnect costs at most
  ten of them;
- resuming is exact — the seeds depend only on `(base_seed, rep)`, so
  replication 137 gives the same numbers whenever it runs;
- cells whose final table already exists are skipped.

**After a disconnect: re-run cells 1 and 4.** Nothing else. The printout will
say `[done]` for finished cells and `[resume]` for the interrupted one.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE        = "/content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision"
CODE_DIR    = f"{BASE}/code"
RESULTS_DIR = f"{BASE}/results_R300"     # results MUST live on Drive      # must live on Drive, not /content

import sys, os
sys.path.insert(0, CODE_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

!pip install -q xgboost pyarrow 2>/dev/null | tail -1

import numpy, pandas, sklearn
print("numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| sklearn", sklearn.__version__)
print("code    ->", CODE_DIR)
print("results ->", RESULTS_DIR)
print("files   :", sorted(f for f in os.listdir(CODE_DIR) if f.endswith('.py')))

Mounted at /content/drive
numpy 2.0.2 | pandas 2.2.2 | sklearn 1.6.1
code    -> /content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision/code
results -> /content/drive/MyDrive/Conformal_Prediction_Research/Class_Conditional_CP_with_Data_Augmentation/csda_revision/results_R300
files   : ['calibration.py', 'clustering_utils.py', 'ding_conformal_utils.py', 'groups.py', 'metrics.py', 'run_grid.py', 'scores.py', 'simulate.py']


In [ ]:
import shutil; shutil.rmtree(RESULTS_DIR + "_validation", ignore_errors=True)

In [ ]:
# Verify the modules on Drive are the current ones before committing hours to a run.
import inspect, simulate as S, calibration as C, metrics as M

checks = {
    "simulate: checkpointing":      "checkpoint_every" in inspect.signature(S.run).parameters,
    "simulate: separate eps_tr":    "rng_gtr" in inspect.getsource(S.run_replication),
    "calibration: warning filter":  "empty slice" in inspect.getsource(C._silent),
    "metrics: binned spread":       "n_bins" in inspect.signature(M.rare_class_spread).parameters,
    "metrics: paired_delta":        hasattr(M, "paired_delta"),
    "metrics: load skips ckpt":     "__ckpt" in inspect.getsource(M.load),
}
for k, v in checks.items():
    print(("  ok  " if v else " STALE") + " | " + k)
if not all(checks.values()):
    raise SystemExit("re-upload the stale modules to Drive before running")

  ok   | simulate: checkpointing
  ok   | simulate: separate eps_tr
  ok   | calibration: warning filter
  ok   | metrics: binned spread
  ok   | metrics: paired_delta
  ok   | metrics: load skips ckpt


## 2. Configuration and cost

`strict=True` here: the per-replication assertions check that the splits are
disjoint, that the balanced test set really is balanced, that every class is
present in training, and that the group acts only on the nuisance block. They
are worth their cost on three replications and are turned off for the campaign.

Validation output goes to a separate directory so it cannot be mistaken for
campaign results.

In [ ]:
import shutil
shutil.rmtree(RESULTS_DIR + "_validation", ignore_errors=True)

In [ ]:
from simulate import SimConfig, run, _population_size
from dataclasses import replace
import time

BASE_CFG = SimConfig(
    scenario="s3_nuisance", model="XGBoost",
    n_informative=5, n_nuisance=20, class_sep=0.8, balanced=False,
    n_cal=500, n_train=4000,
    n_test_pop=2000, n_test_balanced_per_class=500,
    n_transforms=16, alpha=0.10,
    run_train_aug=True, strict=True,
    outdir=RESULTS_DIR + "_validation",
)

print("E[N_cal,rare] :", BASE_CFG.expected_rare_cal())
print("population    :", f"{_population_size(BASE_CFG):,}", "observations")

t = time.time()
run(replace(BASE_CFG, n_reps=3), verbose=True)
per_rep = (time.time() - t) / 3

print(f"\n{per_rep:.1f}s per replication")
print(f"R=300, one cell        : {per_rep*300/60:.0f} min")
print(f"R=300, 6 cells (a=0.10): {per_rep*300*6/3600:.1f} h")
print(f"R=300, 12 cells (both) : {per_rep*300*12/3600:.1f} h")

E[N_cal,rare] : 20.0
population    : 38,000 observations
[done] s3_nuisance_XGBoost_ncal00500_a01_B16_exact already complete

0.3s per replication
R=300, one cell        : 2 min
R=300, 6 cells (a=0.10): 0.2 h
R=300, 12 cells (both) : 0.3 h


### Read that number before going on

If one cell exceeds ~90 minutes, the session will drop mid-cell. That is
survivable — the checkpoint resumes — but it means babysitting. Two ways to
shorten it, in order of preference:

- `n_test_pop=2000`. Only the marginal metrics use that set, and 2000 points
  already give a standard error near 0.007 on marginal coverage. Per-class
  metrics are unaffected.
- `R=200`. The pilot put the requirement at ~100 replications for the classwise
  arms and ~250 for the naive one; 200 covers everything except the naive arm's
  worst case, and that arm is a diagnostic, not a headline result.

Do **not** shrink `n_test_balanced_per_class`: it sets the precision of the
per-class coverage estimates, which is where the paper's claims live.

## 3. What runs

Six calibration sizes, chosen so that `E[N_cal,rare]` lands on
5, 10, 20, 50, 100, 200 — bracketing the `1/alpha = 10` boundary where the
classwise quantile stops existing.

Ordered smallest first: those cells are cheaper *and* carry the regime result,
so an interrupted session leaves the most useful tables already on Drive.

`alpha=0.05` is a second pass. It moves the boundary to 20 and shows the
practical rule scales with alpha, but it is a reinforcement, not the main
result. Add it once the first pass is complete.

In [ ]:
R      = 300
NCAL   = [125, 250, 500, 1250, 2500, 5000]   # E[N_rare] = 5, 10, 20, 50, 100, 200
ALPHAS = [0.05] #[0.10]                              # add 0.05 for the second pass

todo = [(n, a) for a in ALPHAS for n in NCAL]
for n, a in todo:
    c = replace(BASE_CFG, n_cal=n, alpha=a)
    print(f"n_cal={n:>5}  alpha={a}  E[N_rare]={c.expected_rare_cal():>5.0f}  "
          f"population={_population_size(c):>8,}")
print(f"\n{len(todo)} cells at R={R}")

n_cal=  125  alpha=0.05  E[N_rare]=    5  population=  37,250
n_cal=  250  alpha=0.05  E[N_rare]=   10  population=  37,500
n_cal=  500  alpha=0.05  E[N_rare]=   20  population=  38,000
n_cal= 1250  alpha=0.05  E[N_rare]=   50  population=  39,500
n_cal= 2500  alpha=0.05  E[N_rare]=  100  population=  42,000
n_cal= 5000  alpha=0.05  E[N_rare]=  200  population=  47,000

6 cells at R=300


## 4. Run

Re-runnable. Finished cells are skipped, interrupted ones resume from their
checkpoint. The `try/except` stops one failed cell from killing the rest — read
the traceback, fix it, re-run this cell.

In [ ]:
import traceback

for n_cal, alpha in todo:
    cfg = replace(BASE_CFG, n_cal=n_cal, alpha=alpha, n_reps=R,
                  strict=False, outdir=RESULTS_DIR)
    print(f"\n=== n_cal={n_cal}  alpha={alpha}  "
          f"E[N_rare]={cfg.expected_rare_cal():.0f}", flush=True)
    try:
        run(cfg, verbose=True, checkpoint_every=10)
    except Exception:
        traceback.print_exc()
        print("--- cell failed, continuing with the next one", flush=True)

print("\n=== pass complete")


=== n_cal=125  alpha=0.05  E[N_rare]=5
[resume] s3_nuisance_XGBoost_ncal00125_a005_B16_exact from replication 140
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 150/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 160/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 170/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 180/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 190/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 200/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 210/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 220/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 230/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 240/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 250/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 260/300 (checkpoint)
[s3_nuisance_XGBoost_ncal00125_a005_B16_exact] 270/300 (checkpoint)
[

## 5. Progress\n\nWhat is on Drive, and how far the unfinished cell got.

In [ ]:
import glob, pandas as pd
from simulate import _find, _read

rows = []
for n_cal, alpha in todo:
    cfg  = replace(BASE_CFG, n_cal=n_cal, alpha=alpha, n_reps=R, outdir=RESULTS_DIR)
    stem = os.path.join(RESULTS_DIR, f"{cfg.tag()}__long")
    final, ckpt = _find(stem), _find(stem + "__ckpt")
    if final:
        status, done = "complete", R
    elif ckpt:
        status, done = "in progress", int(_read(ckpt)["rep"].max()) + 1
    else:
        status, done = "not started", 0
    rows.append({"n_cal": n_cal, "alpha": alpha,
                 "E_rare": cfg.expected_rare_cal(),
                 "status": status, "reps_done": done, "of": R})

prog = pd.DataFrame(rows)
display(prog)
print(f"{(prog.status == 'complete').sum()}/{len(prog)} cells complete "
      f"| {prog.reps_done.sum()}/{R*len(prog)} replications")

size_mb = sum(os.path.getsize(p) for p in glob.glob(f"{RESULTS_DIR}/*")) / 1e6
print(f"{size_mb:.1f} MB on Drive")

,n_cal,alpha,E_rare,status,reps_done,of
0,125,0.1,5.0,complete,300,300
1,250,0.1,10.0,complete,300,300
2,500,0.1,20.0,complete,300,300
3,1250,0.1,50.0,complete,300,300
4,2500,0.1,100.0,complete,300,300
5,5000,0.1,200.0,complete,300,300


6/6 cells complete | 1800/1800 replications
1.1 MB on Drive


## 6. Sanity check on the output

Two things worth catching before the analysis notebook: that the realised
rare-class counts bracket the boundary as intended, and that the marginal arms
sit at the nominal level on the population test set.

In [ ]:
import metrics as M

try:
    df = M.load(RESULTS_DIR)
except FileNotFoundError:
    print("nothing complete yet")
else:
    chk = (df.groupby("n_cal")
             .agg(E_rare=("n_cal_expected_rarest", "first"),
                  realised_min=("n_cal_realized_rarest", "min"),
                  realised_median=("n_cal_realized_rarest", "median"),
                  realised_max=("n_cal_realized_rarest", "max"),
                  reps=("rep", "nunique")).round(1))
    display(chk)

    summ = M.summarize(M.per_replication(df))
    marg = summ[summ.guarantee == "marginal"]
    print(f"marginal arms, CovMarginal: {marg.CovMarginal.min():.3f} - "
          f"{marg.CovMarginal.max():.3f}  (nominal {1-BASE_CFG.alpha:.2f})")
    print(f"{len(df):,} long rows across {df.n_cal.nunique()} calibration sizes")

nothing complete yet


## Next

Point `RESULTS_DIR` in `analysis.ipynb` at this directory.

For the second pass, set `ALPHAS = [0.05]` in cell 3 and re-run cells 3 and 4:
the `alpha=0.10` cells are already on disk and will be skipped.

The control cells — the other three scenarios and the other two models at a
single calibration size — are a separate pass. Change `scenario`, the
`n_informative`/`n_nuisance`/`class_sep`/`balanced` block, and `model` in
`BASE_CFG`, then set `NCAL = [500]`.